In [1]:
import sys
!{sys.executable} -m pip install "accelerate>=1.1.0"
!pip install -U sentence-transformers
!pip install -U accelerate
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
# import tensorflow_hub as hub
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (classification_report, f1_score, precision_score,
                             recall_score, accuracy_score)
from sklearn.preprocessing import normalize
import warnings
warnings.filterwarnings('ignore')

# sentence-based embedding approaches for Twitter Climate Change Sentiment Dataset
# NOTE: takes about 1.5-2 minutes to run (depending on some parameter changes)

# NOTE: Used ChatGPT/Google Gemini to assist with writing code for this step
# (INTERSPERSE THROUGHOUT)
df = pd.read_csv("twitter_sentiment_data.csv")
df = df[df["sentiment"] != 2]
texts = df["message"].astype(str).tolist()
y = df["sentiment"]
print("Twitter Climate Change Sentiment Dataset (after removing News data):", df.shape)
# df_filtered = df[df["sentiment"] != 2]
# texts_filtered = df_filtered["message"].astype(str).tolist()
# y_filtered = df_filtered["sentiment"]
# print(df_filtered.shape)

# APPROACH 1 - SentenceBERT: SBERT generates embeddings using Siamese networks
# to compare pairs of sentences
sbert = SentenceTransformer("all-MiniLM-L6-v2")
X_sbert = sbert.encode(texts, batch_size = 32, show_progress_bar = True)
X_sbert = normalize(X_sbert)

# APPROACH 2 - SentenceBERT + contrastive pairs: fine-tune SBERT on sarcastic vs.
# non-sarcastic tweet pairs (iSarcasm or SARC dataset) before using it for stance

def build_isarcasm_pairs(df, num_samples = 8000):
    isarcasm_pairs = []
    df = df.dropna(subset = ["tweet", "rephrase"]).reset_index(drop = True)
    for i in range(min(num_samples, len(df))):
        tweet = str(df.iloc[i]["tweet"])
        rephrase = str(df.iloc[i]["rephrase"])
        # negative pair - different sarcasm labels (same meaning: sarcastic vs. literal)
        isarcasm_pairs.append(InputExample(texts = [tweet, rephrase], label = 0.0))
        # positive pair - same sarcasm labels (identical meaning)
        isarcasm_pairs.append(InputExample(texts = [tweet, tweet], label = 1.0))
        isarcasm_pairs.append(InputExample(texts = [rephrase, rephrase], label = 1.0))
    random.shuffle(isarcasm_pairs)
    return isarcasm_pairs

isarcasm_df = pd.read_csv("train.En.csv")
print("iSarcasm Dataset:", isarcasm_df.shape)
train_examples = build_isarcasm_pairs(isarcasm_df, num_samples = 8000)
train_data_loader = DataLoader(train_examples, shuffle = True, batch_size = 16)
sbert_fine_tuned = SentenceTransformer("all-MiniLM-L6-v2")
# train_loss = losses.CosineSimilarityLoss(sbert_fine_tuned)
train_loss = losses.MultipleNegativesRankingLoss(sbert_fine_tuned)
sbert_fine_tuned.fit(train_objectives = [(train_data_loader, train_loss)],
                     epochs = 3, show_progress_bar = True,
                     warmup_steps = int(len(train_data_loader) * 0.1))
X_sbert_fine_tuned = sbert_fine_tuned.encode(texts, batch_size = 32, show_progress_bar = True)
X_sbert_fine_tuned = normalize(X_sbert_fine_tuned)

# APPROACH 3 - Universal Sentence Encoder (USE): USE generates embeddings using a
# Transformer architecture or Deep Averaging Networks (DAN)
# use = hub.load("https://tfhub.dev/google/universal-sentence-encoder/4")
# X_use = use(df["message"].tolist()).numpy()

# classification of all 3 approaches using logistic regression (stable + strong)
def evaluate_approach(X, y, name):
    classifier = LogisticRegression(max_iter = 2000, class_weight = "balanced")
    y_pred = cross_val_predict(classifier, X, y, cv = 5)
    print(f"\n=== {name} ===")
    print("Macro-F1:", np.round(f1_score(y, y_pred, average = "macro"), 4))
    print("Weighted-F1:", np.round(f1_score(y, y_pred, average = "weighted"), 4))
    print("Macro-Precision:", np.round(precision_score(y, y_pred, average = "macro"), 4))
    print("Macro-Recall:", np.round(recall_score(y, y_pred, average = "macro"), 4))
    print("Accuracy:", np.round(accuracy_score(y, y_pred), 4))
    print("Metrics by class:\n", classification_report(y, y_pred))

evaluate_approach(X_sbert, y, "SentenceBERT (SBERT)")
evaluate_approach(X_sbert_fine_tuned, y, "SentenceBERT (SBERT) + Contrastive Pairs "
                                         "[iSarcasm Dataset Fine-Tuning]")
# evaluate_approach(X_use, y, "Universal Sentence Encoder (USE)")



[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: /usr/local/bin/python3.13 -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Twitter Climate Change Sentiment Dataset (after removing News data): (34667, 3)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12469.36it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 1084/1084 [00:26<00:00, 41.16it/s]


iSarcasm Dataset: (3468, 10)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14923.26it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss


Batches: 100%|██████████| 1084/1084 [00:28<00:00, 38.34it/s]



=== SentenceBERT (SBERT) ===
Macro-F1: 0.5713
Weighted-F1: 0.6581
Macro-Precision: 0.5582
Macro-Recall: 0.6345
Accuracy: 0.636
Metrics by class:
               precision    recall  f1-score   support

          -1       0.34      0.67      0.45      3990
           0       0.46      0.59      0.52      7715
           1       0.87      0.65      0.74     22962

    accuracy                           0.64     34667
   macro avg       0.56      0.63      0.57     34667
weighted avg       0.72      0.64      0.66     34667


=== SentenceBERT (SBERT) + Contrastive Pairs [iSarcasm Dataset Fine-Tuning] ===
Macro-F1: 0.5745
Weighted-F1: 0.6638
Macro-Precision: 0.5612
Macro-Recall: 0.6352
Accuracy: 0.6421
Metrics by class:
               precision    recall  f1-score   support

          -1       0.34      0.66      0.45      3990
           0       0.48      0.59      0.53      7715
           1       0.87      0.66      0.75     22962

    accuracy                           0.64     34667
 